In [1]:
!pip install mlflow

Defaulting to user installation because normal site-packages is not writeable


In [2]:
# Import 
import pandas as pd
from sklearn.model_selection import train_test_split
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import os

In [3]:
#Load the data 
df = pd.read_parquet('D:\\Projects\\Customer_Retention\\data\\02_rfm_features.parquet', engine='pyarrow')
df.head()

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Segment,RFM_Score,Segment_Label
0,12346,326,12,77556.46,2,5,5,255,12,Loyal Customers
1,12347,2,8,4921.53,5,4,5,545,14,Champions
2,12348,75,5,2019.40,3,4,4,344,11,Loyal Customers
3,12349,19,4,4428.69,5,3,5,535,13,Champions
4,12350,310,1,334.40,2,1,2,212,5,At Risk


In [4]:
#defining churn based on recency 
CHURN_THRESHOLD = 90

df['Churned'] = (df['Recency'] > CHURN_THRESHOLD).astype(int)

In [5]:
churn_counts = df['Churned'].value_counts()
churn_percentages = df["Churned"].value_counts(normalize=True) * 100

In [6]:
print("Count of customers:\n", churn_counts, "\n")
print("Percentage of customers:\n", churn_percentages.round(2))

Count of customers:
 Churned
1    2989
0    2889
Name: count, dtype: int64 

Percentage of customers:
 Churned
1    50.85
0    49.15
Name: proportion, dtype: float64


In [7]:
# Spliting the datasets for model training and testing the model

features = ['Frequency', 'Monetary', 'F_Score', 'M_Score']
X = df[features]
y = df['Churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
print(f"Total customers: {len(df)}")
print(f"Training set size: {len(X_train)} customers")
print(f"Testing set size: {len(X_test)} customers")

Total customers: 5878
Training set size: 4702 customers
Testing set size: 1176 customers


In [ ]:
db_path = r"D:\\Projects\\Customer_Retention\\mlflow.db"
tracking_uri = f"sqlite:///{db_path.replace('\\', '/')}"
mlflow.set_tracking_uri(tracking_uri)

In [10]:
experiment_name = "Customer_Churn_Prediction"
mlflow.set_experiment(experiment_name)
print("MLflow is successfully connected!")
print(f"Database located at: {mlflow.get_tracking_uri()}")

MLflow is successfully connected!
Database located at: sqlite:///C:\Users\viraj\mlflow.db


In [11]:
with mlflow.start_run(run_name="Logistic_Regression_Baseline"):
    

    lr_model = LogisticRegression(max_iter=1000, random_state=42)
    lr_model.fit(X_train, y_train)

    y_pred = lr_model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("model_type", "Logistic Regression")

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    mlflow.sklearn.log_model(lr_model, "model")
    
    print("Model trained and logged to MLflow!")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")

2026/03/26 16:37:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/26 16:37:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


KeyboardInterrupt: 